# Publication notes
See [the project README](../README.md) and [setup instructions](../docs/setup.md). Outputs were cleared for publication; this file has not been rerun during packaging. Notebook 6 is the current integration development reference; supplied-run evidence and limitations are documented in the README.


## Cell 1 — Load stuff

In [ ]:
import os, sys, time, json, inspect, subprocess, hashlib
from pathlib import Path
from dataclasses import dataclass
from functools import partial
import numpy as np
import matplotlib.pyplot as plt



REPO = Path(os.environ.get('ADAPTIVE_SLS_REPO', str(Path.home() / 'projects/adaptive_sls_complete'))).expanduser().resolve()
required = ['src/gpu_sls/generic_mpc_wrapper.py', 'src/gpu_sls/sqp.py',
            'src/linearization_sls/test_path.py', 'experiments/quadrotor/quadrotor_common.py']
if not all((REPO / p).is_file() for p in required):
    raise FileNotFoundError(f'Set REPO to the NEW trustworthyrobotics/adaptive_sls checkout. Current: {REPO}')
os.environ['JAX_PLATFORMS'] = os.environ.get('NB06_PLATFORM', 'cpu')  # use cuda, never gpu
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
sys.path.insert(0, str(REPO / 'src'))
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import gpu_sls
from gpu_sls.gpu_sls import (SLSConfig, observable_subspace_factorizer,
    adaptive_posterior_factors, girard_reduction_opt, posterior_trace_information_gain)
from gpu_sls.sqp import SQPConfig
from gpu_sls.gpu_admm import ADMMConfig
from gpu_sls.generic_mpc_wrapper import GenericMPCControllerWrapper



if not Path(gpu_sls.__file__).resolve().is_relative_to(REPO / 'src'):
    raise RuntimeError('Wrong gpu_sls module already loaded. Restart kernel and run Cell 1 first.')
if 'enable_information_cost' not in SLSConfig.__dataclass_fields__:
    raise RuntimeError('Selected repository lacks the required information-cost interface.')
try:
    SOURCE_COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
except (OSError, subprocess.CalledProcessError):
    SOURCE_COMMIT = 'archive: no git metadata'
SOURCE_HASHES = {p: hashlib.sha256((REPO/p).read_bytes()).hexdigest() for p in required}
print('Repository:', REPO)
print('Commit:', SOURCE_COMMIT)
print('Solver:', inspect.getfile(GenericMPCControllerWrapper))
print('JAX:', jax.__version__, '| devices:', jax.devices())
print('If immrax/trajax is missing, use the dependency setup for this repository; no fake solver fallback is used.')

## Cell 2 — Define all initial parameters and constants

In [ ]:
DT, KA, WIDTH, O_ORIGIN, LENGTH = 0.02, 1.0, 0.10, 0.20, 0.01
MASS_PRIOR = np.array([0.005, 0.030])
alpha_from_mass = lambda m: KA*DT**2 / (np.asarray(m) + KA*DT**2)
ALPHA_LO, ALPHA_HI = alpha_from_mass(MASS_PRIOR[::-1])
ALPHA_C, ALPHA_R = (ALPHA_LO+ALPHA_HI)/2, (ALPHA_HI-ALPHA_LO)/2
SIGMA_M = 0.5e-3
SIGMA = SIGMA_M / LENGTH
PROCESS_BOUND_M = np.array([5e-6, 5e-6, 0.0])
SENSOR_BOUND_M = 0.0
assert SENSOR_BOUND_M == 0.0, 'Noisy-state MPC is not implemented here. Keep exact observations for this milestone.'
NX, NT, NU, NA = 3, 1, 1, 4
HORIZON, EXECUTION_STEPS, QMAX = 6, 6, 8
INFO_WEIGHT = 3.0
MOVE_MAX, BACKWARD_ALLOWANCE = 8e-3/LENGTH, 0.2e-3/LENGTH
DU_MAX, U_MIN, U_MAX = 12e-3/LENGTH, -2e-3/LENGTH, 35e-3/LENGTH
A_MIN = 1e-3/LENGTH
FORCE_MAX, FORCE_SCALE = 25e-3, 20e-3
MOVE_SCALE, COMMAND_SCALE = 2e-3/LENGTH, 10e-3/LENGTH
W_COST = jnp.array([1.0, 0.25, 0.05])
GAIN_REG = 1e-7
ENABLE_LEB = False
RUN_CLOSED_LOOP = True
RUN_COMPARISON = True
REQUIRE_ADMM_CONVERGED = True
CHECK_TOL = 2e-5  # scaled position tolerance: 0.2 micrometers at L=10 mm
F_TEST = jnp.diag(jnp.asarray(PROCESS_BOUND_M / LENGTH))
C_MEAS = jnp.diag(jnp.array([0.0, 1.0, 0.0]))  # mass estimation uses object position only
INITIAL_PHYS = jnp.array([-0.005, 0.0, -0.005])
INITIAL_THETA = jnp.array([0.0])
INITIAL_G = jnp.zeros((1, QMAX)).at[0, 0].set(1.0)
OBSTACLES = jnp.empty((0, 3))

def mass_from_alpha(alpha):
    return KA*DT**2 * (1/np.asarray(alpha)-1)

def parameter_interval(theta_hat, G):
    radius = float(np.abs(np.asarray(G)).sum())
    return float(np.asarray(theta_hat).reshape(-1)[0]) + np.array([-radius, radius])

def reported_mass_interval(theta_hat, G):
    # Reporting intersects the initial physical prior. It does NOT alter the estimator or planner.
    lo, hi = np.maximum(parameter_interval(theta_hat,G)[0], -1), np.minimum(parameter_interval(theta_hat,G)[1], 1)
    if lo > hi:
        raise RuntimeError('Parameter set has empty intersection with the initial prior.')
    return mass_from_alpha(ALPHA_C + ALPHA_R*np.array([hi,lo]))

print('Mass prior [g]:', MASS_PRIOR*1e3)
print('Nominal mass [g]:', mass_from_alpha(ALPHA_C)*1e3)
print('Process bounds [um]:', PROCESS_BOUND_M*1e6)
print('Sensor noise: zero; object position only used for parameter update.')
print('Horizon:', HORIZON, '| execution steps:', EXECUTION_STEPS)
print('Smooth surrogate sigma [mm]:', SIGMA_M*1e3, '| LEB enabled:', ENABLE_LEB)

## Cell 3 — Define simplified smooth model for both prediction and execution/observation

In [ ]:
def positive_smooth(a): #Define a smooth approximation to pushing excitation
    return 0.5*(a + jnp.sqrt(a*a + 4*SIGMA*SIGMA))

def physical_step(z, u, theta): #Define one nominal pusher transition
    alpha = ALPHA_C + ALPHA_R*jnp.ravel(theta)[0]
    g = positive_smooth(u[0]-z[1])
    return jnp.stack([u[0]-(1-alpha)*g, z[1]+alpha*g, u[0]])

def dynamics(s, u, t, *, parameter): #Adapt the model to the controller’s required interface
    # DT is already fixed in alpha; parameter keeps the repository's required signature.
    del t, parameter
    return jnp.concatenate([physical_step(s[:NX],u,s[NX:]), s[NX:]])

def disturbance(X):
    return jnp.broadcast_to(F_TEST, (X.shape[0], NX, NX))

def nominal_force(z, u, theta): #Define predicted pushing force
    alpha = ALPHA_C + ALPHA_R*jnp.ravel(theta)[0]
    return KA*LENGTH*(1-alpha)*positive_smooth(u[0]-z[1])

def simulate_step(z, u, hidden_theta, xi): #Define simulated plant execution
    xi = np.asarray(xi, dtype=float)
    assert xi.shape == (NX,) and np.max(np.abs(xi)) <= 1+1e-12
    nominal = physical_step(z,u,jnp.array([hidden_theta]))
    next_z = nominal + F_TEST @ jnp.asarray(xi)
    return next_z, np.asarray(next_z-nominal)

## Cell 4 — Define disturbance cost and constraints

In [ ]:
FORCE_NOISE_MARGIN = KA*PROCESS_BOUND_M[0]
G_FORCE_CAP = (FORCE_MAX-FORCE_NOISE_MARGIN)/(KA*LENGTH*(1-ALPHA_LO))
A_FORCE_CAP = G_FORCE_CAP - SIGMA**2/G_FORCE_CAP

def base_cost(W, reference, s, u, t):
    object_penalty = W[0]*((s[1]-reference[t,1])/MOVE_SCALE)**2
    force_proxy = KA*LENGTH*(1-ALPHA_LO)*positive_smooth(u[0]-s[1])
    stage = object_penalty + W[1]*(force_proxy/FORCE_SCALE)**2 + W[2]*((u[0]-s[2])/COMMAND_SCALE)**2
    # Small numerical regularization, not a target for the unknown parameter.
    stage += 1e-6*(jnp.dot(s,s)+jnp.dot(u,u))
    terminal = object_penalty + 1e-6*jnp.dot(s,s)
    return jnp.where(t < HORIZON, stage/HORIZON, terminal/HORIZON)

def constraints(s, u, t):
    a, du = u[0]-s[1], u[0]-s[2]
    state = jnp.array([s[1]-MOVE_MAX, -s[1]-BACKWARD_ALLOWANCE])
    control = jnp.array([U_MIN-u[0], u[0]-U_MAX, du-DU_MAX,
                         -du-DU_MAX, A_MIN-a, a-A_FORCE_CAP])
    control = jnp.where(t < HORIZON, control, -jnp.ones_like(control))
    return jnp.concatenate([state,control])

NC = 8 #number of constraints

def nominal_rollout(s0, U):
    def step(s, pair):
        k,u=pair
        nxt=dynamics(s,u,k,parameter=DT)
        return nxt,nxt
    _, following = jax.lax.scan(step,s0,(jnp.arange(U.shape[0]),U))
    return jnp.concatenate([s0[None,:],following],axis=0)

def make_seed(s0, excitation=5e-3/LENGTH):
    s=s0; commands=[]
    for k in range(HORIZON):
        u=jnp.array([s[1]+excitation])
        commands.append(u)
        s=dynamics(s,u,k,parameter=DT)
    U=jnp.stack(commands)
    return nominal_rollout(s0,U), U

def stage_residuals(X,U):
    U_pad=jnp.concatenate([U,jnp.zeros((1,NU))],axis=0)
    return np.asarray(jax.vmap(constraints)(X,U_pad,jnp.arange(HORIZON+1)))

## Cell 5 — Define the estimator and interval check

In [ ]:
@jax.jit
def estimator_update(s_previous, u, observed_next, G):
    theta=s_previous[NX:]
    predictor=lambda candidate: physical_step(s_previous[:NX],u,candidate)
    prediction=predictor(theta)
    E_theta=jax.jacfwd(predictor)(theta)
    P_obs, Pi, W_obs=observable_subspace_factorizer(E_theta,F_TEST,C_MEAS)
    _, Lgain, G_pre, _=adaptive_posterior_factors(
        G,Pi,W_obs,gain_regularization=GAIN_REG,enable_posterior_gate=True)
    residual=observed_next-prediction
    innovation=P_obs@residual
    theta_next=theta+Lgain@innovation
    G_next=girard_reduction_opt(G_pre,QMAX)
    updated=jnp.concatenate([observed_next,theta_next])
    return updated,G_next,dict(E=E_theta,W=W_obs,L=Lgain,residual=residual,
                               innovation=innovation,Pi=Pi)

def independent_interval_update(interval,z,u,next_z):
    g=float(positive_smooth(u[0]-z[1]))
    H=ALPHA_R*g
    r=float(next_z[1]-z[1])-ALPHA_C*g
    if H <= 1e-12:
        return interval.copy()
    b=PROCESS_BOUND_M[1]/LENGTH
    updated=np.array([max(interval[0],(r-b)/H), min(interval[1],(r+b)/H)])
    if updated[0]>updated[1]+1e-10:
        raise RuntimeError('Empty independent interval: data contradict the assumed model/bound.')
    return updated

## Cell 6 — Check model/estimator interfaces

In [ ]:
s0=jnp.concatenate([INITIAL_PHYS,INITIAL_THETA]); u0=jnp.array([0.5])
theta_fixture=float((alpha_from_mass(0.015)-ALPHA_C)/ALPHA_R)
next_fixture,_=simulate_step(INITIAL_PHYS,u0,theta_fixture,[0.4,-0.6,0.0])
s1,G1,audit=estimator_update(s0,u0,next_fixture,INITIAL_G)
affine_prediction=physical_step(INITIAL_PHYS,u0,INITIAL_THETA)+audit['E'][:,0]*theta_fixture
assert np.allclose(affine_prediction,physical_step(INITIAL_PHYS,u0,jnp.array([theta_fixture])),atol=1e-12)
assert disturbance(s0[None,:]).shape==(1,NX,NX)
assert constraints(s0,u0,0).shape==(NC,)
lo,hi=parameter_interval(s1[NX:],G1)
assert lo-1e-10 <= theta_fixture <= hi+1e-10
X_seed,U_seed=make_seed(s0)
assert np.max(stage_residuals(X_seed,U_seed)) <= CHECK_TOL
print('Interface checks passed. Fixture mass interval [g]:',reported_mass_interval(s1[NX:],G1)*1e3)
print('State / generator / disturbance shapes:',s0.shape,INITIAL_G.shape,disturbance(X_seed).shape)
print('Nominal seed maximum constraint residual:',np.max(stage_residuals(X_seed,U_seed)))

## Cell 7 — Construct the complete adaptive MPC controller

In [ ]:
@dataclass
class PusherConfig:
    n: int=NA
    nu: int=NU
    N: int=HORIZON
    W: object=None
    u_ref: object=None
    dt: float=DT

DIAGNOSTIC_NAMES=['sls_converged','sls_residual','admm_converged','admm_iterations',
    'admm_primal_residual','admm_dual_residual','admm_primal_tolerance','admm_dual_tolerance',
    'admm_primal_worst_flat_index','admm_primal_worst_z','admm_primal_worst_w']

def build_controller(info_weight=INFO_WEIGHT):
    start=jnp.concatenate([INITIAL_PHYS,INITIAL_THETA])
    X,U=make_seed(start)
    cfg=PusherConfig(W=W_COST,u_ref=jnp.array([0.0]))
    sls=SLSConfig(max_sls_iterations=4,sls_primal_tol=1e-4,enable_fastsls=True,
        warm_start=False,n_x=NX,n_w=NX,n_theta=NT,num_param=NT,q_max=QMAX,adaptive=True,
        enable_linearization_error=ENABLE_LEB,enable_disturbance_variation_error=False,
        enable_posterior_gate=True,enable_information_cost=info_weight>0,
        information_cost_weight=float(info_weight),information_cost_discount=1.0,
        information_gain_regularization=GAIN_REG)
    controller=GenericMPCControllerWrapper(sls,
        SQPConfig(max_sqp_iterations=8,warm_start=False,feas_tol=1e-6,step_tol=1e-6,line_search=True),
        ADMMConfig(max_iterations=800,eps_abs=1e-6,eps_rel=1e-5,initial_rho=10.0,
                   rho_max=1e6,rho_update_frequency=25),
        config=cfg,dynamics=dynamics,constraints=constraints,obstacles=OBSTACLES,
        cost=base_cost,num_constraints=NC,disturbance=disturbance,X_in=X,U_in=U,
        limited_memory=False,shift=1,measurement_matrix=C_MEAS,
        Q_bar=jnp.diag(jnp.array([1e-3,1.0,1e-3,1e-3])),
        R_bar=jnp.array([[0.1]]),Q_f_bar=jnp.diag(jnp.array([1e-3,1.0,1e-3,1e-3])))
    compiled=controller._solve
    def recorded_solve(*args,**kwargs):
        result=compiled(*args,**kwargs)
        if len(result)!=28:
            raise RuntimeError('Adaptive solver return layout changed; inspect the new source before continuing.')
        controller.notebook_diagnostics=dict(zip(DIAGNOSTIC_NAMES,result[17:]))
        return result
    controller._solve=recorded_solve
    controller.notebook_diagnostics={}
    return controller

def solve_plan(controller,s,G):
    reference=jnp.zeros((HORIZON+1,NA)).at[:,NX:].set(s[NX:])
    # Anchor at actual observed state; retain shifted previous command seed.
    controller.X0=nominal_rollout(s,controller.U0)
    start=time.perf_counter()
    out=controller.run(x0=s,G_0=G,reference=reference,nominal_param=s[NX:],parameter=DT)
    out[0].block_until_ready()
    elapsed=time.perf_counter()-start
    u,X,U,V,backoffs,Phi_x,Phi_u,K,EN,KN,GN,P_inv=out
    diag={k:np.asarray(v).item() for k,v in controller.notebook_diagnostics.items()}
    if not all(np.isfinite(np.asarray(a)).all() for a in [u,X,U,backoffs,GN]):
        raise FloatingPointError(f'Nonfinite solver output. Diagnostics: {diag}')
    rollout=nominal_rollout(s,U)
    dynamics_defect=float(np.max(np.abs(np.asarray(X-rollout))))
    nominal_violation=float(np.max(stage_residuals(rollout,U)))
    tightened_violation=float(np.max(stage_residuals(X,U)+np.asarray(backoffs)))
    diag.update(seconds=elapsed,dynamics_defect=dynamics_defect,
                nominal_violation=nominal_violation,tightened_violation=tightened_violation)
    return dict(u=u,X=X,U=U,backoffs=backoffs,GN=GN,diagnostics=diag)

## Cell 8 — Independent one-step execution guard and plant-independent planner call

In [ ]:
def first_step_guard(s,G,u):
    ti=parameter_interval(s[NX:],G)
    lo,hi=max(ti[0],-1.0),min(ti[1],1.0)
    if lo>hi:
        return False,{'reason':'empty parameter-prior intersection'}
    alpha_lo,alpha_hi=ALPHA_C+ALPHA_R*np.array([lo,hi])
    a=float(u[0]-s[1]); g=float(positive_smooth(a))
    du=float(u[0]-s[2])
    object_lo=float(s[1])+alpha_lo*g-PROCESS_BOUND_M[1]/LENGTH
    object_hi=float(s[1])+alpha_hi*g+PROCESS_BOUND_M[1]/LENGTH
    force_hi=KA*LENGTH*(1-alpha_lo)*g+FORCE_NOISE_MARGIN
    residuals=np.array([U_MIN-float(u[0]),float(u[0])-U_MAX,abs(du)-DU_MAX,
        A_MIN-a,object_hi-MOVE_MAX,-object_lo-BACKWARD_ALLOWANCE,
        (force_hi-FORCE_MAX)/(KA*LENGTH)])
    return bool(np.max(residuals)<=CHECK_TOL),dict(max_residual=float(np.max(residuals)),
        force_upper_N=force_hi,object_lower=object_lo,object_upper=object_hi)

controller_demo = build_controller(info_weight=3.0)

print("Information reward enabled:",
      controller_demo.sls_config.enable_information_cost)
print("Information weight:",
      controller_demo.sls_config.information_cost_weight)

print('Compiling and solving one adaptive plan. This first call may take several minutes...',flush=True)
PLAN=solve_plan(controller_demo,jnp.concatenate([INITIAL_PHYS,INITIAL_THETA]),INITIAL_G)
print('Diagnostics:',PLAN['diagnostics'])
ok,guard=first_step_guard(jnp.concatenate([INITIAL_PHYS,INITIAL_THETA]),INITIAL_G,PLAN['u'])
print('First-step guard:',ok,guard)
print('Six candidate commands, offsets from original contact boundary [mm]:',np.asarray(PLAN['U']).ravel()*LENGTH*1e3)
print('Nominal predicted total object displacement [mm]:',np.asarray(PLAN['X'])[:,1]*LENGTH*1e3)

## Cell 9 — Complete plan–execute–observe–estimate loop

In [ ]:
def run_experiment(hidden_mass=0.015,seed=7,info_weight=INFO_WEIGHT,steps=EXECUTION_STEPS):
    if not MASS_PRIOR[0]<=hidden_mass<=MASS_PRIOR[1]:
        raise ValueError('Evaluation mass must lie inside the stated prior.')
    hidden_theta=float((alpha_from_mass(hidden_mass)-ALPHA_C)/ALPHA_R)
    noise=np.random.default_rng(seed).uniform(-1,1,(steps,NX)); noise[:,2]=0
    controller=build_controller(info_weight)
    s=jnp.concatenate([INITIAL_PHYS,INITIAL_THETA]); G=INITIAL_G
    interval=np.array([-1.0,1.0]); rows=[]; plans=[]
    states=[np.asarray(s)]; generators=[np.asarray(G)]; intervals=[reported_mass_interval(s[NX:],G)]
    status='completed'
    for k in range(steps):
        print(f'Planning step {k+1}/{steps}...',flush=True)
        plan=solve_plan(controller,s,G); diag=plan['diagnostics']
        ok,guard=first_step_guard(s,G,plan['u'])
        print(f"  solve={diag['seconds']:.2f}s, ADMM={diag['admm_converged']}, SLS={diag['sls_converged']}, guard={ok}",flush=True)
        if not ok or (REQUIRE_ADMM_CONVERGED and not diag['admm_converged']):
            status='stopped_before_execution: guard or ADMM check failed'
            plans.append(plan)
            print(status,diag,guard)
            break
        old_s=s; u=plan['u']
        next_z,d=simulate_step(s[:NX],u,hidden_theta,noise[k])
        interval=independent_interval_update(interval,s[:NX],u,next_z)
        s,G,est=estimator_update(s,u,next_z,G)
        mi=reported_mass_interval(s[NX:],G)
        theta_interval=parameter_interval(s[NX:],G)
        coverage=theta_interval[0]-1e-9<=hidden_theta<=theta_interval[1]+1e-9
        residual_model=np.asarray(next_z-physical_step(old_s[:NX],u,jnp.array([hidden_theta])))
        bounded=bool(np.all(np.abs(residual_model)<=PROCESS_BOUND_M/LENGTH+1e-12))
        force=KA*LENGTH*float(u[0]-next_z[0])
        motion=float(next_z[1])*LENGTH
        actual_limits=bool(motion<=MOVE_MAX*LENGTH+CHECK_TOL*LENGTH and
                           motion>=-BACKWARD_ALLOWANCE*LENGTH-CHECK_TOL*LENGTH and
                           force<=FORCE_MAX+KA*LENGTH*CHECK_TOL)
        estimate_mass=float(mass_from_alpha(ALPHA_C+ALPHA_R*float(s[NX])))
        rows.append(dict(step=k+1,command_m=float(u[0])*LENGTH+O_ORIGIN-WIDTH,
            object_travel_mm=motion*1e3,force_mN=force*1e3,
            mass_estimate_g=estimate_mass*1e3,mass_lo_g=mi[0]*1e3,mass_hi_g=mi[1]*1e3,
            width_g=(mi[1]-mi[0])*1e3,covered=bool(coverage),bound_ok=bounded,
            limits_ok=actual_limits,solve_seconds=diag['seconds'],
            independent_theta_width=float(interval[1]-interval[0]),
            posterior_radius=float(np.abs(np.asarray(G)).sum()),
            admm_converged=bool(diag['admm_converged']),sls_converged=bool(diag['sls_converged'])))
        states.append(np.asarray(s)); generators.append(np.asarray(G)); intervals.append(mi); plans.append(plan)
        print(f'  travel={motion*1e3:.4f} mm, mass interval={mi*1e3} g, covered={coverage}',flush=True)
        if not (coverage and bounded and actual_limits):
            status='stopped_after_execution: coverage, model bound, or physical limit check failed'
            print(status)
            break
    return dict(rows=rows,states=np.asarray(states),generators=np.asarray(generators),
                mass_intervals=np.asarray(intervals),plans=plans,status=status,
                hidden_mass=hidden_mass,seed=seed,info_weight=info_weight,requested_steps=steps)

RESULT = (
    run_experiment(info_weight=3.0)
    if RUN_CLOSED_LOOP else None
)
if RESULT is None:
    print('Closed-loop run disabled; set RUN_CLOSED_LOOP=True and rerun this cell.')

## Cell 10 — Result

In [ ]:
if RESULT is not None:
    rows=RESULT['rows']
    print('Status:',RESULT['status'],'| completed:',len(rows),'/',RESULT['requested_steps'])
    if rows:
        for r in rows:
            print(f"{r['step']:2d}: move={r['object_travel_mm']:.4f} mm | F={r['force_mN']:.3f} mN | "
                  f"mass=[{r['mass_lo_g']:.3f},{r['mass_hi_g']:.3f}] g | "
                  f"coverage={r['covered']} limits={r['limits_ok']}")
        steps=np.arange(len(rows)+1); states=RESULT['states']; mi=RESULT['mass_intervals']*1e3
        fig,ax=plt.subplots(2,2,figsize=(11,7))
        ax[0,0].plot(steps*DT,states[:,1]*LENGTH*1e3,'o-',label='Executed object')
        ax[0,0].axhline(MOVE_MAX*LENGTH*1e3,color='r',ls='--',label='Total movement limit')
        ax[0,0].set(xlabel='Time [s]',ylabel='Motion from original position [mm]');ax[0,0].legend()
        ax[0,1].fill_between(steps,mi[:,0],mi[:,1],alpha=.25,label='Reported feasible mass interval')
        ax[0,1].plot(steps,mass_from_alpha(ALPHA_C+ALPHA_R*states[:,NX])*1e3,'o-',label='Adaptive center converted to mass')
        ax[0,1].axhline(RESULT['hidden_mass']*1e3,color='k',ls='--',label='Truth: evaluation only')
        ax[0,1].set(xlabel='Executed transitions',ylabel='Mass [g]');ax[0,1].legend(fontsize=8)
        ax[1,0].plot(steps,np.diff(mi,axis=1).ravel(),'o-')
        ax[1,0].set(xlabel='Executed transitions',ylabel='Mass interval width [g]')
        ax[1,1].step(np.arange(len(rows))*DT,[r['force_mN'] for r in rows],where='post',label='Synthetic contact force')
        ax[1,1].axhline(FORCE_MAX*1e3,color='r',ls='--',label='Force limit')
        ax[1,1].set(xlabel='Time [s]',ylabel='Force [mN]');ax[1,1].legend()
        for a in ax.ravel():a.grid(alpha=.3)
        fig.tight_layout();plt.show()
        print('Coverage / model-bound / limit failures:',
              sum(not r['covered'] for r in rows),sum(not r['bound_ok'] for r in rows),sum(not r['limits_ok'] for r in rows))
        print('This establishes integration only if the requested steps complete and diagnostics are acceptable.')

## Cell 11 — Save

In [ ]:
SAVE_RESULTS=False
if SAVE_RESULTS and RESULT is not None:
    project=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
    destination=project/'results'/('nb06_'+time.strftime('%Y%m%d_%H%M%S'))
    destination.mkdir(parents=True,exist_ok=False)
    np.savez(destination/'rollout.npz',states=RESULT['states'],generators=RESULT['generators'],
             mass_intervals=RESULT['mass_intervals'])
    metadata=dict(status=RESULT['status'],rows=RESULT['rows'],source_commit=SOURCE_COMMIT,
        source_hashes=SOURCE_HASHES,hidden_mass=RESULT['hidden_mass'],seed=RESULT['seed'],
        sigma_m=SIGMA_M,process_bound_m=PROCESS_BOUND_M.tolist(),sensor_bound_m=SENSOR_BOUND_M,
        enable_linearization_error=ENABLE_LEB,information_weight=INFO_WEIGHT,
        model='fixed-length smooth positive-part surrogate; not finite-kappa Drake',
        diagnostics=[p['diagnostics'] for p in RESULT['plans']])
    (destination/'metadata.json').write_text(json.dumps(metadata,indent=2))
    print('Saved:',destination)
else:
    print('Set SAVE_RESULTS=True in this cell to export this run.')